In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
from torch.amp import autocast, GradScaler

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This notebook requires a GPU.")

device = torch.device("cuda")
print(f"Using device: {device}")

In [ ]:
MODEL_STRING = "distilbert-base-cased"

In [ ]:
pretrained_model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_STRING,
    num_labels=1,
)

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_STRING)

In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd

class SentimentReviewDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        if hasattr(dataframe, 'toPandas'):
            self.data = dataframe.toPandas()
        else:
            self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        required_cols = ['review_text', 'rating_normalized']
        missing_cols = [col for col in required_cols if col not in self.data.columns]
        if missing_cols:
            raise ValueError(f"DataFrame missing required columns: {missing_cols}")
        
        # Drop rows with null values in critical columns
        self.data = self.data.dropna(subset=['review_text', 'rating_normalized'])
        
        # Reset index after filtering
        self.data = self.data.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Get review text and sentiment target
        review_text = str(row['review_text'])
        sentiment_target = float(row['rating_normalized'])
        # Tokenize the review text
        encoding = self.tokenizer(
            review_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["rating_normalized"] = torch.tensor(sentiment_target, dtype=torch.float)
        
        return item

In [ ]:
data_dir = os.path.abspath("parquet")

# Load only the columns needed for training (column pruning!)
columns_needed = ["review_text", "rating_normalized", "source"]

train_df = pd.read_parquet(
    f"{data_dir}/train.parquet",
    columns=columns_needed  # Only read these columns!
)

val_df = pd.read_parquet(
    f"{data_dir}/val.parquet",
    columns=columns_needed
)

test_df = pd.read_parquet(
    f"{data_dir}/test.parquet",
    columns=columns_needed
)

#Create datasets
train_dataset = SentimentReviewDataset(
    dataframe=train_df,
    tokenizer=tokenizer,
    max_length=512
)

val_dataset = SentimentReviewDataset(
    dataframe=val_df,
    tokenizer=tokenizer,
    max_length=512
)

test_dataset = SentimentReviewDataset(
    dataframe=test_df,
    tokenizer=tokenizer,
    max_length=512
)

train_loader = DataLoader(
    train_dataset,
    batch_size=384,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=384,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=384,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

In [ ]:
def model_accuracy(model, data_loader):
  model.eval()
  total, absolute_errors = 0, 0
  with torch.no_grad():
    for batch in data_loader:
        # batch["input_ids"] = batch["input_ids"].to(device)
        # batch["attention_mask"] = batch["attention_mask"].to(device)
        # batch["rating_normalized"] = batch["rating_normalized"].to(device)
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model.to(device)(**batch)
        predicted = outputs.logits.squeeze(-1)
        labels = batch["rating_normalized"]

        total += labels.size(0)
        absolute_errors += torch.abs(predicted - labels).sum().item()
  return absolute_errors/total

In [ ]:
# to make faster
scaler = GradScaler(device='cuda')

# Load Base model and wrap into a classifier
finetuned_model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_STRING,
    num_labels=1,
).to(device)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    finetuned_model = nn.DataParallel(finetuned_model)

optimizer = torch.optim.AdamW(finetuned_model.parameters(), lr = .00001)
criterion = nn.MSELoss()

EPOCHS = 3
epoch_train_mae = []
epoch_val_mae = []

epoch_train_mae.append(model_accuracy(finetuned_model, train_loader))
epoch_val_mae.append(model_accuracy(finetuned_model, val_loader))

for epoch in range(EPOCHS):
  finetuned_model.train()
  epoch_loss = 0

  for batch in train_loader:
    # batch["input_ids"] = batch["input_ids"].to(device)
    # batch["attention_mask"] = batch["attention_mask"].to(device)
    # batch["rating_normalized"] = batch["rating_normalized"].to(device)
    batch = {k: v.to(device) for k, v in batch.items()}

    optimizer.zero_grad()

    # This speeds up on colab
    with autocast(device_type="cuda"):
      outputs = finetuned_model(**batch)
      pred = outputs.logits.squeeze(-1)
      labels = batch["rating_normalized"]
      loss = criterion(pred, labels)
    scaler.scale(loss).backward()
    torch.nn.utils.clip_grad_norm_(finetuned_model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
      
    epoch_loss += loss.item()

  avg_loss = epoch_loss / len(train_loader)
  print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss/len(train_loader):.4f}")

  epoch_train_mae.append(model_accuracy(finetuned_model, train_loader))
  epoch_val_mae.append(model_accuracy(finetuned_model, val_loader))

In [ ]:
save_dir = os.path.abspath("models/finetuned_distilbert")

model_to_save = finetuned_model.module if isinstance(finetuned_model, nn.DataParallel) else finetuned_model
model_to_save.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Model saved to {save_dir}")

In [ ]:
print("untrained MAE:",  model_accuracy(pretrained_model, test_loader))
print("finetuned MAE:",  model_accuracy(finetuned_model, test_loader))

In [ ]:
# A review of The Machinist from someone which gave 3.5 stars -> 7/10
review = "I don’t know if it’s because I’ve seen too many movies with this twist now but I kind of saw it coming from like 10 minutes in. It didn’t really matter though because it was still really well done. I think that the whole theme of guilt eating away at the rest of your life was great. Along with the depiction of psychosis/schizophrenia (I’m not sure which one it would technically be) was top notch, which probably has a lot to do with the actor. Christian Bales performance overall was just amazing, the transformation he went through to is unreal. He is easily one of the best actors of our time, and definitely the most devoted. This movie is worth a watch for Bale alone, and the rest of the movie is just a nice little bonus."

encoding = tokenizer(
    review,
    max_length=256,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

finetuned_model.eval()
with torch.no_grad():
    outputs = finetuned_model(**encoding.to(device))
    prediction = outputs.logits.squeeze().item()

prediction_0_10 = torch.sigmoid(torch.tensor(prediction)).item() * 10
print("Mapped to [0,10]:", prediction_0_10)

In [ ]:
# diplay test results:
finetuned_model.eval()
predicted = []
labels = []
with torch.no_grad():
  for batch in test_loader:
      # batch["input_ids"] = batch["input_ids"].to(device)
      # batch["attention_mask"] = batch["attention_mask"].to(device)
      # batch["rating_normalized"] = batch["rating_normalized"].to(device)
      batch = {k: v.to(device) for k, v in batch.items()}

      outputs = finetuned_model.to(device)(**batch)
      predicted += outputs.logits.squeeze(-1).cpu()
      labels += batch["rating_normalized"].cpu()

In [ ]:
plt.scatter(labels, predicted)

plt.title("Finetuned Predicted vs True Labels")
y=plt.ylim(0,1)
plt.xlabel("True Labels")
plt.ylabel("Predicted Labels")
plt.grid(True)

plt.show()

In [ ]:
from scipy.stats import norm

def error_stats(predicted, labels):
    pred = (torch.stack(predicted) if isinstance(predicted, list) else predicted) * 10
    true = (torch.stack(labels) if isinstance(labels, list) else labels) * 10
    errors = (pred - true).numpy()
    return errors, errors.mean(), errors.std()

finetuned_errors, finetuned_mean, finetuned_std = error_stats(predicted, labels)

# print(f"Pretrained: mean error = {untrained_mean:+.3f}; std = {untrained_std:.3f}")
print(f"Finetuned: mean error = {finetuned_mean:+.3f}; std = {finetuned_std:.3f}")

for errs, mu, sigma, title in [
    (finetuned_errors, finetuned_mean, finetuned_std, "Finetuned"),
]:
    plt.hist(errs, bins=40, density=True, alpha=0.6)
    xs = np.linspace(errs.min(), errs.max(), 200)
    plt.plot(xs, norm.pdf(xs, mu, sigma))
    plt.title(f"{title}: mean={mu:+.2f}, std={sigma:.2f}")
    plt.show()